In [7]:
import sys
sys.path.insert(1, '/home/bricej/MyPythonLibrary/StageM2_IVT/library/')
import domain
from domain import *
from climbas import *
import numpy as np
import geopandas as gpd
import xarray as xr
import pandas as pd
from netCDF4 import Dataset
import dask
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

# Period of study & Domains

In [8]:
iyear = 2000
fyear = 2019
domains = ['Eastern Hindu Kush', 
            'Western Himalaya', 
            'Eastern Himalaya',
            'Central Himalaya', 
            'Karakoram', 
            'Western Pamir', 
            'Pamir Alay', 
            'Northern/Western Tien Shan', 
            'Dzhungarsky Alatau', 
            'Western Kunlun Shan', 
            'Nyainqentanglha', 
            'Gangdise Mountains', 
            'Hengduan Shan', 
            'Tibetan Interior Mountains', 
            'Tanggula Shan', 
            'Eastern Tibetan Mountains', 
            'Qilian Shan', 
            'Eastern Kunlun Shan', 
            'Altun Shan', 
            'Eastern Tien Shan', 
            'Central Tien Shan', 
            'Eastern Pamir']

# Data from ERA5

In [9]:
variables = ['t2m','FLH','tcwv','ivt','vimd','tp','sf','rf']

data = {}

path = '/bettik/PROJECTS/pr-regional-climate/bricej/era5/monthly/'

files = {'t2m': 't2m_monthly_era5_1940-2025_NH_025x025.nc',
        'FLH': 'FLH_monthly_era5_1940-2025_MA_025x025.nc',
        'tcwv': 'tcwv_monthly_era5_1940-2025_NH_025x025.nc',
        'ivt': 'ivt_monthly_era5_1940-2025_MA_025x025.nc',
        'vimd': 'vimd_monthly_era5_1940-2025_NH_025x025.nc',
        'tp': 'tp_MA_1940-2025_ERA5.nc',
        'sf': 'sf_MA_1940-2025_ERA5.nc',
        'rf': 'rf_MA_1940-2025_ERA5.nc'}

## atmospheric variables

In [4]:
variables_atm = ['t2m','FLH','tcwv','ivt','vimd']

files = {'t2m': 't2m_monthly_era5_1940-2025_NH_025x025.nc',
        'FLH': 'FLH_monthly_era5_1940-2025_MA_025x025.nc',
        'tcwv': 'tcwv_monthly_era5_1940-2025_NH_025x025.nc',
        'ivt': 'ivt_monthly_era5_1940-2025_MA_025x025.nc',
        'vimd': 'vimd_monthly_era5_1940-2025_NH_025x025.nc'}

for var in variables_atm:
    data[var] = {}
    data_var = xr.open_dataset(path + files[var])[var]
    if var in ['t2m', 'tcwv', 'ivt', 'vimd']:
        data_var = data_var.rename({'valid_time': 'time','latitude':'lat','longitude':'lon'})
    data_var = data_var.sel(time=slice(str(iyear),str(fyear)))
    data_var = field_dom(data_var, 'HMA')
    
    for dom in domains:
        data[var][dom] = himap(data_var, dom)

## precipitation variables

In [ ]:
variables_pre = ['tp','sf','rf']

files = {'tp': 'tp_MA_1940-2025_ERA5.nc',
        'sf': 'sf_MA_1940-2025_ERA5.nc',
        'rf': 'rf_MA_1940-2025_ERA5.nc'}

for var in variables_pre:
    data[var] = {}
    data_var = xr.open_dataset(path + files[var])[var]
    data_var = data_var.sel(time=slice(str(iyear),str(fyear)))
    data_var = field_dom(data_var, 'HMA')
    
    for dom in domains:
        data[var][dom] = himap(data_var, dom)

# Trends 2000-2019 per variable

## parameters

In [11]:
iyear = 2000
fyear = 2019
domains = ['Eastern Hindu Kush', 
            'Western Himalaya', 
            'Eastern Himalaya',
            'Central Himalaya', 
            'Karakoram', 
            'Western Pamir', 
            'Pamir Alay', 
            'Northern/Western Tien Shan', 
            'Dzhungarsky Alatau', 
            'Western Kunlun Shan', 
            'Nyainqentanglha', 
            'Gangdise Mountains', 
            'Hengduan Shan', 
            'Tibetan Interior Mountains', 
            'Tanggula Shan', 
            'Eastern Tibetan Mountains', 
            'Qilian Shan', 
            'Eastern Kunlun Shan', 
            'Altun Shan', 
            'Eastern Tien Shan', 
            'Central Tien Shan', 
            'Eastern Pamir']

files = {'t2m': 't2m_monthly_era5_1940-2025_NH_025x025.nc',
        'FLH': 'FLH_monthly_era5_1940-2025_MA_025x025.nc',
        'tcwv': 'tcwv_monthly_era5_1940-2025_NH_025x025.nc',
        'ivt': 'ivt_monthly_era5_1940-2025_MA_025x025.nc',
        'vimd': 'vimd_monthly_era5_1940-2025_NH_025x025.nc',
        'tp': 'tp_MA_1940-2025_ERA5.nc',
        'sf': 'sf_MA_1940-2025_ERA5.nc',
        'rf': 'rf_MA_1940-2025_ERA5.nc'}

var = 't2m'
file = files[var]

## data

In [16]:
data = {}

data_var = xr.open_dataset(path + file)[var]
if var in ['t2m', 'tcwv', 'ivt', 'vimd']:
    data_var = data_var.rename({'valid_time': 'time','latitude':'lat','longitude':'lon'})
data_var = data_var.sel(time=slice(str(iyear),str(fyear)))
data_var = field_dom(data_var, 'HMA')
    
for dom in domains:
    data[dom] = himap(data_var, dom)

## annual selection

In [22]:
data_annual = {}

for dom in domains:
    data_annual[dom] = data[dom].groupby('time.year').mean(dim='time', skipna=True)
    data_annual[dom] = data_annual[dom].mean(dim=['lat','lon'], skipna=True)

## trend

In [24]:
trend = {}

for dom in domains:
    trend[dom] = stats.linregress(
                data_annual[dom].year.values,
                data_annual[dom].values
            )

In [27]:
slope = {}
pvalue = {}

for dom in domains:
    slope[dom] = trend[dom][0]
    pvalue[dom] = trend[dom][3]

In [30]:
rows = []

for region, res in trend.items():
    if isinstance(res, float) and np.isnan(res):
        # cas où tu avais mis NaN (pas assez de données)
        rows.append({
            "region": region,
            "slope": np.nan,
            "intercept": np.nan,
            "rvalue": np.nan,
            "pvalue": np.nan,
            "stderr": np.nan
        })
    else:
        rows.append({
            "region": region,
            "slope": res.slope,
            "intercept": res.intercept,
            "rvalue": res.rvalue,
            "pvalue": res.pvalue,
            "stderr": res.stderr
        })

df = pd.DataFrame(rows)

In [32]:
path = '/bettik/PROJECTS/pr-regional-climate/bricej/paper/'
filename = f"{var}_era5_trends_HIMAP_regions_{iyear}-{fyear}.csv"

df.to_csv(path + filename, index=False)